# 01. Eksplorasi Data Kurs IDR/USD Mingguan

Notebook ini menjawab pertanyaan pertama skripsi: **bagaimana karakteristik data kurs rupiah terhadap dolar AS** pada periode minggu pertama Oktober 2021 sampai minggu pertama Oktober 2023. Data diambil dari Lampiran 1 skripsi (sumber asli id.investing.com).

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from kurs_forecast.data import load_kurs, split_train_test
serie = load_kurs(); train, test = split_train_test(serie)
print(f"Total {len(serie)} minggu: training {len(train)}, testing {len(test)}")

Total 105 minggu: training 84, testing 21


## Statistik deskriptif
Kurs bergerak di kisaran Rp14.000 sampai Rp15.800 dengan tren naik (rupiah melemah) sepanjang periode, sejalan dengan kenaikan suku bunga The Fed.

In [2]:
serie.describe().round(2).to_frame().T

,count,mean,std,min,25%,50%,75%,max
kurs,105.0,14871.15,451.04,14070.0,14385.0,14910.0,15220.0,15735.0


In [3]:
from kurs_forecast.plots import plot_series
plot_series(train, test);

## Uji linieritas
Skripsi memakai uji Terasvirta di R (chi-square 6,104; p = 0,047). Di sini dipakai versi regresi Volterra yang prinsipnya sama: apakah suku kuadrat dan kubik dari prediksi linier menambah daya jelas secara signifikan.

In [4]:
from kurs_forecast.diagnostics import terasvirta_proxy, adf_test, kpss_test
terasvirta_proxy(serie.values)

{'f_statistic': 2.992825430248506,
 'pvalue': 0.05465708191579351,
 'nonlinier': False}

## Stasioneritas
Selain plot ACF/PACF seperti skripsi, dipakai uji ADF (H0: tidak stasioner) dan KPSS (H0: stasioner) agar keputusan tidak hanya bergantung pada pembacaan visual.

In [5]:
from kurs_forecast.transform import estimate_lambda, boxcox_transform, difference, standardize
lam = estimate_lambda(train)
z, _, _ = standardize(boxcox_transform(train.values, lam))
print('lambda Box-Cox =', round(lam, 4))
pd.DataFrame({
    'level': adf_test(train.values), 'boxcox': adf_test(z),
    'diff1': adf_test(difference(z, 1)), 'diff2': adf_test(difference(z, 2))}).T[['statistic', 'pvalue', 'stasioner']]

lambda Box-Cox = -1.0


,statistic,pvalue,stasioner
level,-1.536141,0.515563,False
boxcox,-1.536412,0.515428,False
diff1,-5.652831,0.000001,True
diff2,-5.815228,0.0,True


In [6]:
from kurs_forecast.plots import plot_acf_pacf
plot_acf_pacf(train.values, 'training (level)');
plot_acf_pacf(difference(z, 1), 'Box-Cox, differencing 1');
plot_acf_pacf(difference(z, 2), 'Box-Cox, differencing 2');

## Kesimpulan EDA
- Data level tidak stasioner (ADF p > 0,05; ACF turun lambat), konsisten dengan skripsi.
- Lambda Box-Cox pada batas -1, sama seperti skripsi (-0,9999), sehingga transformasi yang dipakai adalah 1/Z.
- Setelah differencing 2 kali data stasioner, dan ACF memotong di lag 1 serta PACF di lag 1 sampai 2, sehingga kandidat model adalah ARIMA (0,2,1), (1,2,0), (2,2,0), (1,2,1), dan (2,2,1).